In [ ]:
###1234فوق تحت

import gradio as gr
from PIL import Image, ImageDraw, ImageFont
import re
import os
import requests
import tempfile

# روابط الخطوط من Supabase
SUPABASE_FONTS = {
    "AdobeArabic-Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/AdobeArabic-Regular.otf",
    "GE SS Two Light": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/GE%20SS%20Two%20Light.otf",
    "Montserrat-Medium": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Montserrat-Medium.ttf",
    "Montserrat-SemiBold": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Montserrat-SemiBold.otf",
    "MyriadPro-Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/MyriadPro-Regular.otf",
    "Ping LCG Bold": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Ping%20LCG%20Bold.otf",
    "Ping LCG Light": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Ping%20LCG%20Light.otf",
    "Ping LCG Medium": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Ping%20LCG%20Medium.otf",
    "Ping LCG Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Ping%20LCG%20Regular.otf",
    "PingAR+LT-Bold": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Bold.otf",
    "PingAR+LT-Heavy": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Heavy.otf",
    "PingAR+LT-Light": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Light.otf",
    "PingAR+LT-Medium": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Medium.otf",
    "PingAR+LT-Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Regular.otf",
    "PingAR+LT-Thin": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/PingAR+LT-Thin.otf",
    "Poppins-ExtraLight": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Poppins-ExtraLight.ttf",
    "Poppins-Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/Poppins-Regular.ttf",
    "TheYearofTheCamel-ExtraBold": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/TheYearofTheCamel-ExtraBold.otf",
    "TheYearofTheCamel-Light": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/TheYearofTheCamel-Light.otf",
    "TheYearofTheCamel-Regular": "https://berljkcatnnorgljhepe.supabase.co/storage/v1/object/public/fonts/TheYearofTheCamel-Regular.otf"
}

# مجلد مؤقت لحفظ الخطوط
TEMP_FONTS_DIR = tempfile.mkdtemp(prefix="fonts_")

def download_fonts():
    """تحميل جميع الخطوط من Supabase إلى المجلد المؤقت"""
    downloaded_fonts = {}

    print("📥 جاري تحميل الخطوط من Supabase...")

    for font_name, font_url in SUPABASE_FONTS.items():
        try:
            print(f"⬇️  جاري تحميل: {font_name}")
            response = requests.get(font_url, timeout=30)
            response.raise_for_status()

            # حفظ الخط في ملف مؤقت
            file_extension = font_url.split('.')[-1]
            filename = f"{font_name}.{file_extension}"
            filepath = os.path.join(TEMP_FONTS_DIR, filename)

            with open(filepath, 'wb') as f:
                f.write(response.content)

            downloaded_fonts[font_name] = filepath
            print(f"✅ تم تحميل: {font_name}")

        except Exception as e:
            print(f"❌ خطأ في تحميل {font_name}: {e}")

    print(f"🎉 تم تحميل {len(downloaded_fonts)} من أصل {len(SUPABASE_FONTS)} خط")
    return downloaded_fonts

def get_available_fonts():
    """الحصول على قائمة الخطوط المتاحة"""
    downloaded_fonts = download_fonts()

    if downloaded_fonts:
        font_names = list(downloaded_fonts.keys())
        print(f"📋 الخطوط المتاحة: {len(font_names)} خط")
        return sorted(font_names)
    else:
        print("⚠️ لم يتم تحميل أي خطوط، سيتم استخدام الخطوط النظامية")
        return ["استخدام الخطوط النظامية"]

def load_selected_font(font_name, size):
    """تحميل الخط المحدد من الملفات المؤقتة"""
    if not font_name or font_name == "استخدام الخطوط النظامية":
        return load_system_font(size)

    try:
        # البحث عن الملف المناسب للخط
        for filename in os.listdir(TEMP_FONTS_DIR):
            if filename.startswith(font_name):
                font_path = os.path.join(TEMP_FONTS_DIR, filename)
                if os.path.exists(font_path):
                    print(f"📖 جاري تحميل الخط: {font_name} بحجم {size}")
                    font = ImageFont.truetype(font_path, int(size))
                    print(f"✅ تم تحميل الخط بنجاح: {font_name}")
                    return font

        print(f"❌ لم يتم العثور على ملف للخط: {font_name}")
        return load_system_font(size)

    except Exception as e:
        print(f"❌ خطأ في تحميل الخط {font_name}: {e}")
        return load_system_font(size)

def load_system_font(size):
    """الخطوط الاحتياطية للنظام"""
    candidates = [
        "/usr/share/fonts/truetype/noto/NotoNaskhArabic-Regular.ttf",
        "/usr/share/fonts/truetype/noto/NotoSansArabic-Regular.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ]
    for p in candidates:
        try:
            if os.path.exists(p):
                return ImageFont.truetype(p, int(size))
        except:
            continue

    return ImageFont.load_default()

# باقي الدوال تبقى كما هي بدون تغيير
def ensure_rgba(img):
    return img.convert("RGBA") if img and img.mode != "RGBA" else img

def place_xy(pos, W, H, w, h, margin):
    if pos == "أعلى يمين":
        return W - w - margin, margin
    if pos == "أعلى يسار":
        return margin, margin
    if pos == "أسفل يمين":
        return W - w - margin, H - h - margin
    if pos == "أسفل يسار":
        return margin, H - h - margin
    if pos == "المنتصف":
        return (W - w) // 2, (H - h) // 2
    return margin, margin

def is_arabic_char(char):
    return 0x0600 <= ord(char) <= 0x06FF

def is_symbol_char(char):
    symbols = "!@#$%^&*()_+-=[]{}|\\~`;:'\",.<>/?؟"
    return char in symbols

def is_english_char(char):
    return char.isascii() and char.isalpha()

def reverse_text_inside_brackets(text):
    def reverse_match(match):
        open_bracket = match.group(1)
        content = match.group(2)
        close_bracket = match.group(3)
        words = content.split()
        reversed_words = ' '.join(reversed(words))
        return open_bracket + reversed_words + close_bracket

    pattern = r'([\(\[\{<])([^\(\)\[\]\{\}<>]*)([\)\]\}>])'
    return re.sub(pattern, reverse_match, text)

def flip_brackets(text):
    bracket_map = {
        '(': ')', ')': '(',
        '[': ']', ']': '[',
        '{': '}', '}': '{',
        '<': '>', '>': '<'
    }

    def replacer(match):
        inner = match.group(2)
        open_b, close_b = match.group(1), match.group(3)
        return bracket_map.get(close_b, close_b) + inner + bracket_map.get(open_b, open_b)

    pattern = r'([\(\[\{<])(.*?)([\)\]\}>])'
    return re.sub(pattern, replacer, text)

def preprocess_punctuation(text):
    punctuation_marks = ";:'\",.<>/?؟"
    result, i = "", 0
    while i < len(text):
        char = text[i]
        if char in punctuation_marks:
            if char == '.' and i > 0 and i < len(text) - 1:
                prev_char, next_char = text[i-1], text[i+1]
                if prev_char.isdigit() and next_char.isdigit():
                    result += char
                    i += 1
                    continue
            if i > 0 and text[i-1] != ' ':
                result += ' '
            result += char
            if i < len(text) - 1 and text[i+1] != ' ':
                result += ' '
        else:
            result += char
        i += 1
    return result

def protect_bracket_content(text):
    pattern = r'([\(\[\{<][^()\[\]{}<>]+[\)\]\}>])'
    matches = re.findall(pattern, text)
    protected = text
    for m in matches:
        new_m = m.replace(" ", "\u2423")
        protected = protected.replace(m, new_m)
    return protected

def segment_text(text):
    segments, current_segment, current_type = [], "", None
    for char in text:
        char_type = (
            'space' if char.isspace()
            else 'arabic' if is_arabic_char(char)
            else 'symbol' if is_symbol_char(char)
            else 'number' if char.isdigit()
            else 'english' if is_english_char(char)
            else 'other'
        )
        if current_type is None:
            current_type, current_segment = char_type, char
        elif char_type == current_type or (char_type == 'space' and current_type == 'space'):
            current_segment += char
        else:
            segments.append((current_segment, current_type))
            current_segment, current_type = char, char_type
    if current_segment:
        segments.append((current_segment, current_type))
    return segments

def get_font_for_segment(segment_type, arabic_font, symbol_font, default_font):
    if segment_type == 'arabic':
        return arabic_font
    elif segment_type == 'symbol':
        return symbol_font
    else:
        return symbol_font

def get_rendered_width(draw, segments, arabic_font, symbol_font, default_font):
    total_width = 0
    for segment, seg_type in segments:
        font_to_use = get_font_for_segment(seg_type, arabic_font, symbol_font, default_font)
        try:
            bbox = draw.textbbox((0, 0), segment, font=font_to_use)
            total_width += bbox[2] - bbox[0]
        except:
            total_width += len(segment) * int(getattr(font_to_use, 'size', 12)) // 2
    return total_width

def split_into_words(segments):
    words, current_word = [], []
    for segment, seg_type in segments:
        if seg_type == 'space':
            if current_word:
                words.append(current_word)
                current_word = []
            words.append([(segment, seg_type)])
        else:
            segment = segment.replace("\u2423", " ")
            current_word.append((segment, seg_type))
    if current_word:
        words.append(current_word)
    return words

def wrap_text_and_reverse_words(text, max_width, draw, arabic_font, symbol_font, default_font):
    paragraphs = text.splitlines() if "\n" in text else [text]
    all_lines = []
    for paragraph in paragraphs:
        if not paragraph.strip():
            all_lines.append([])
            continue
        segments = segment_text(paragraph)
        words = split_into_words(segments)
        current_line_words, current_line_width, i = [], 0, 0
        while i < len(words):
            word = words[i]
            word_width = get_rendered_width(draw, word, arabic_font, symbol_font, default_font)
            if current_line_width + word_width <= max_width:
                current_line_words.append(word)
                current_line_width += word_width
                i += 1
            else:
                if current_line_words:
                    reversed_line = []
                    for word in reversed(current_line_words):
                        reversed_line.extend(word)
                    all_lines.append(reversed_line)
                    current_line_words, current_line_width = [], 0
                else:
                    if word_width > max_width and len(word) > 1:
                        segment_line, segment_line_width = [], 0
                        for segment, seg_type in word:
                            segment_width = get_rendered_width(draw, [(segment, seg_type)], arabic_font, symbol_font, default_font)
                            if segment_line_width + segment_width <= max_width:
                                segment_line.append((segment, seg_type))
                                segment_line_width += segment_width
                            else:
                                if segment_line:
                                    all_lines.append(list(reversed(segment_line)))
                                segment_line = [(segment, seg_type)]
                                segment_line_width = segment_width
                        if segment_line:
                            all_lines.append(list(reversed(segment_line)))
                        i += 1
                    else:
                        all_lines.append(list(reversed(word)))
                        i += 1
        if current_line_words:
            reversed_line = []
            for word in reversed(current_line_words):
                reversed_line.extend(word)
            all_lines.append(reversed_line)
    return all_lines

def compose(base_img, logo_img, text, selected_font, font_size, color, text_pos, margin,
            stroke_w, stroke_color, logo_scale, logo_opacity, logo_pos):
    if base_img is None:
        return None
    base = ensure_rgba(base_img) or base_img.convert("RGBA")
    draw = ImageDraw.Draw(base)

    if logo_img is not None:
        logo = ensure_rgba(logo_img) or logo_img.convert("RGBA")
        target_w = max(1, int(base.width * (logo_scale / 100.0)))
        ratio = target_w / max(1, logo.width)
        target_h = max(1, int(logo.height * ratio))
        logo = logo.resize((target_w, target_h), Image.LANCZOS)
        if "A" not in logo.getbands():
            from PIL import Image as PILImage
            alpha = PILImage.new("L", logo.size, int(255 * logo_opacity))
            logo.putalpha(alpha)
        else:
            a = logo.getchannel("A").point(lambda p: int(p * logo_opacity))
            logo.putalpha(a)
        lx, ly = place_xy(logo_pos, base.width, base.height, logo.width, logo.height, int(margin))
        base.alpha_composite(logo, dest=(lx, ly))

    if text and text.strip():
        cleaned_text = text.replace("\u200f", "").replace("\u200e", "").replace("\ufeff", "").strip()

        reversed_inside_brackets = reverse_text_inside_brackets(cleaned_text)
        preprocessed_text = preprocess_punctuation(reversed_inside_brackets)
        protected_text = protect_bracket_content(preprocessed_text)
        t = flip_brackets(protected_text)

        arabic_font = load_selected_font(selected_font, font_size)
        symbol_font = load_system_font(font_size)
        default_font = ImageFont.load_default()

        m = int(margin)
        max_w = max(1, base.width - 2 * m)
        lines = wrap_text_and_reverse_words(t, max_w, draw, arabic_font, symbol_font, default_font)

        try:
            bbox = draw.textbbox((0, 0), "Ayg", font=arabic_font)
            line_h = int((bbox[3] - bbox[1]) * 1.3)
        except:
            line_h = int(font_size * 1.3)

        max_line_w = 0
        for line_segments in lines:
            if line_segments:
                line_width = get_rendered_width(draw, line_segments, arabic_font, symbol_font, default_font)
                max_line_w = max(max_line_w, line_width)

        text_block_h = line_h * len(lines)
        if text_pos == "أعلى يمين":
            tx, ty, align = base.width - m - max_line_w, m, "right"
        elif text_pos == "أعلى يسار":
            tx, ty, align = m, m, "left"
        elif text_pos == "أسفل يمين":
            tx, ty, align = base.width - m - max_line_w, base.height - m - text_block_h, "right"
        elif text_pos == "أسفل يسار":
            tx, ty, align = m, base.height - m - text_block_h, "left"
        else:
            tx, ty, align = (base.width - max_line_w) // 2, (base.height - text_block_h) // 2, "center"

        def draw_line(x, y, line_segments):
            if not line_segments:
                return
            current_x = x
            line_width = get_rendered_width(draw, line_segments, arabic_font, symbol_font, default_font)
            if align == "right":
                current_x = x + (max_line_w - line_width)
            elif align == "center":
                current_x = x + (max_line_w - line_width) // 2
            for segment, seg_type in line_segments:
                font_to_use = get_font_for_segment(seg_type, arabic_font, symbol_font, default_font)
                segment = segment.replace("\u2423", " ")
                try:
                    draw.text((current_x, y), segment, font=font_to_use, fill=color,
                              stroke_width=int(stroke_w), stroke_fill=stroke_color)
                except:
                    draw.text((current_x, y), segment, font=default_font, fill=color)
                segment_width = get_rendered_width(draw, [(segment, seg_type)], arabic_font, symbol_font, default_font)
                current_x += segment_width

        yy = ty
        for line_segments in lines:
            draw_line(tx, yy, line_segments)
            yy += line_h
    return base

# تشغيل التطبيق
print("🚀 جاري تحضير التطبيق...")
available_fonts = get_available_fonts()

with gr.Blocks(title="كتابة نص عربي - باستخدام خطوطك") as demo:
    gr.Markdown(f"""
    # 📝 كتابة نص عربي - باستخدام خطوطك الخاصة

    **✅ تم تحميل {len(available_fonts)} خط بنجاح**

    **اختر أي خط من القائمة واكتب النص العربي**
    """)

    with gr.Row():
        base_input = gr.Image(type="pil", label="الصورة الأساسية")
        logo_input = gr.Image(type="pil", label="اللوجو (اختياري)")

    text_input = gr.Textbox(label="النص العربي", placeholder="اكتب النص العربي هنا...", lines=4)

    with gr.Row():
        font_choice = gr.Dropdown(
            choices=available_fonts,
            value=available_fonts[0] if available_fonts else "استخدام الخطوط النظامية",
            label=f"اختر الخط ({len(available_fonts)} خط متاح)"
        )
        font_size = gr.Slider(12, 200, value=60, step=1, label="حجم الخط")
        color = gr.ColorPicker(value="#ffffff", label="لون النص")

    with gr.Row():
        stroke_w = gr.Slider(0, 12, value=3, step=1, label="سمك الحد الخارجي")
        stroke_color = gr.ColorPicker(value="#000000", label="لون الحد الخارجي")
        text_pos = gr.Dropdown(
            ["أعلى يمين", "أعلى يسار", "أسفل يمين", "أسفل يسار", "المنتصف"],
            value="أسفل يمين", label="موضع النص"
        )

    with gr.Row():
        margin = gr.Slider(0, 200, value=20, step=1, label="الهامش من الحواف (px)")
        logo_scale = gr.Slider(5, 80, value=25, step=1, label="حجم اللوجو (%)")
        logo_opacity = gr.Slider(0.05, 1.0, value=1.0, step=0.05, label="شفافية اللوجو")

    with gr.Row():
        logo_pos = gr.Dropdown(
            ["أعلى يمين", "أعلى يسار", "أسفل يمين", "أسفل يسار", "المنتصف"],
            value="أعلى يسار", label="موضع اللوجو"
        )

    output = gr.Image(type="pil", label="الصورة الناتجة")

    gr.Button("🖼️ تطبيق النص على الصورة").click(
        compose,
        inputs=[
            base_input, logo_input, text_input, font_choice, font_size, color, text_pos, margin,
            stroke_w, stroke_color, logo_scale, logo_opacity, logo_pos
        ],
        outputs=output
    )

demo.launch()

🚀 جاري تحضير التطبيق...
📥 جاري تحميل الخطوط من Supabase...
⬇️  جاري تحميل: AdobeArabic-Regular
✅ تم تحميل: AdobeArabic-Regular
⬇️  جاري تحميل: GE SS Two Light
✅ تم تحميل: GE SS Two Light
⬇️  جاري تحميل: Montserrat-Medium
✅ تم تحميل: Montserrat-Medium
⬇️  جاري تحميل: Montserrat-SemiBold
✅ تم تحميل: Montserrat-SemiBold
⬇️  جاري تحميل: MyriadPro-Regular
✅ تم تحميل: MyriadPro-Regular
⬇️  جاري تحميل: Ping LCG Bold
✅ تم تحميل: Ping LCG Bold
⬇️  جاري تحميل: Ping LCG Light
✅ تم تحميل: Ping LCG Light
⬇️  جاري تحميل: Ping LCG Medium
✅ تم تحميل: Ping LCG Medium
⬇️  جاري تحميل: Ping LCG Regular
✅ تم تحميل: Ping LCG Regular
⬇️  جاري تحميل: PingAR+LT-Bold
✅ تم تحميل: PingAR+LT-Bold
⬇️  جاري تحميل: PingAR+LT-Heavy
✅ تم تحميل: PingAR+LT-Heavy
⬇️  جاري تحميل: PingAR+LT-Light
✅ تم تحميل: PingAR+LT-Light
⬇️  جاري تحميل: PingAR+LT-Medium
✅ تم تحميل: PingAR+LT-Medium
⬇️  جاري تحميل: PingAR+LT-Regular
✅ تم تحميل: PingAR+LT-Regular
⬇️  جاري تحميل: PingAR+LT-Thin
✅ تم تحميل: PingAR+LT-Thin
⬇️  جاري تحميل: Pop